# Sun Tzu — render final sendiri di Colab T4
**Belum diuji di Colab T4 aktual; perlu QA manusia.**
1. Runtime → Change runtime type → T4 GPU. Tidak membeli kredit/provision otomatis.
2. Isi SHA integrasi lengkap dari pembuat video. Jalankan sel berurutan, **jangan Run all**.
3. Tonton tes 3 detik sebelum menyetujui full render. Chunk 300 frame disimpan di Drive dan diverifikasi saat resume.

Repo privat: Secrets `GITHUB_TOKEN`, fine-grained Contents **Read-only** hanya repo ini, aktifkan Notebook access. Token tidak masuk URL/config/log. Repo publik tidak perlu Secret.
`google-colab-cli` untuk dipasang di mesin lokal Linux/macOS guna mengelola runtime jarak jauh; **tidak diperlukan di dalam notebook ini**.

In [ ]:
BRANCH = 'claude/sun-tzu-latihan-istana'
SNAPSHOT_SHA = ''  # WAJIB SHA integrasi lengkap 40 karakter, bukan SHA preview
USE_GITHUB_SECRET = False
REQUIRE_T4 = True
TEST_START_FRAME = 0  # offset boleh, misalnya 300; harus tersedia 90 frame
CONCURRENCY = 2  # 1 jika OOM; perubahan setting memerlukan tes ulang
DRIVE_BASE = '/content/drive/MyDrive/shorts-render/sun-tzu'


## 1. Checkout snapshot bersih — akses baca saja

In [ ]:
import os, re, json, subprocess, pathlib, tempfile, shutil
assert re.fullmatch(r'[a-f0-9]{40}', SNAPSHOT_SHA), 'Isi SHA integrasi lengkap terlebih dahulu.'
REPO_URL = 'https://github.com/Hapidzfadli/samurai-thunder-shorts.git'
REPO = pathlib.Path('/content/suntzu-repo')
def cmd(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
assert not any(os.environ.get(k) for k in ['GIT_TRACE','GIT_TRACE_CURL','GIT_CURL_VERBOSE']), 'Nonaktifkan git tracing.'
token = None
if USE_GITHUB_SECRET:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    assert token, 'Secret tidak tersedia'
try:
    with tempfile.TemporaryDirectory(prefix='git-readonly-') as td:
        askpass = pathlib.Path(td) / 'askpass.py'
        askpass.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('COLAB_GIT_TOKEN',''))\n")
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0')
        if token: env['COLAB_GIT_TOKEN'] = token
        if not REPO.exists():
            cmd(['git','init',str(REPO)], stdout=subprocess.DEVNULL)
            cmd(['git','-C',str(REPO),'remote','add','origin',REPO_URL])
        git = ['git','-c','credential.helper=','-C',str(REPO)]
        assert subprocess.check_output(git+['remote','get-url','origin'],text=True).strip() == REPO_URL
        assert not subprocess.check_output(git+['status','--porcelain','--untracked-files=no'],text=True).strip(), 'Checkout berubah; jangan menimpa kerja Anda.'
        cmd(git+['fetch','--depth=1','origin','refs/heads/'+BRANCH],env=env)
        cmd(git+['fetch','--depth=1','origin',SNAPSHOT_SHA],env=env)
        cmd(git+['checkout','--detach',SNAPSHOT_SHA],env=env)
        assert subprocess.check_output(git+['rev-parse','HEAD'],text=True).strip()==SNAPSHOT_SHA
finally:
    token = None
    if 'env' in globals(): env.pop('COLAB_GIT_TOKEN', None)
print('Snapshot:', SNAPSHOT_SHA)


## 2. Dependensi terkunci + browser
Colab umumnya menyediakan Node. Jika Node <20, berhenti dan gunakan runtime terbaru atau pasang Node LTS manual. Tidak mengganti lockfile/tidak mengambil Remotion latest.

In [ ]:
node = shutil.which('node')
assert node and int(subprocess.check_output([node,'--version'],text=True).strip().lstrip('v').split('.')[0])>=20, 'Perlu Node >=20.'
cmd(['apt-get','update','-qq'])
packages=['ffmpeg','libnss3','libgbm1','libxkbcommon0','libxcomposite1','libxdamage1','libxfixes3','libxrandr2','libdrm2','libpango-1.0-0','libcairo2','libegl1','libgles2']
for alternatives in [('libatk1.0-0t64','libatk1.0-0'),('libatk-bridge2.0-0t64','libatk-bridge2.0-0'),('libcups2t64','libcups2'),('libasound2t64','libasound2')]:
    selected = None
    for package in alternatives:
        info=subprocess.check_output(['apt-cache','policy',package],text=True)
        if re.search(r'Candidate: (?!\(none\))\S+',info):
            selected=package
            break
    assert selected, 'Missing browser dependency: '+str(alternatives)
    packages.append(selected)
cmd(['apt-get','install','-y','-qq']+packages)
cmd(['npm','ci','--no-audit','--no-fund'],cwd=REPO)
cmd([str(REPO/'node_modules/.bin/remotion'),'browser','ensure'],cwd=REPO)
cmd(['node','--check','scripts/colab_suntzu_render.mjs'],cwd=REPO)


## 3. Drive, konfigurasi, preflight
Setujui akses Drive bila Anda ingin menyimpan chunk/video/log di akun sendiri. Sisakan ruang beberapa kali ukuran final. Jangan menjalankan dua notebook ke folder output sama.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = pathlib.Path(DRIVE_BASE) / SNAPSHOT_SHA
OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG = pathlib.Path('/content/suntzu-config.json')
settings=dict(expectedSha=SNAPSHOT_SHA, outputDir=str(OUTPUT), requireT4=REQUIRE_T4, testStart=TEST_START_FRAME, concurrency=CONCURRENCY, approvedTest=False)
def save_settings(): CONFIG.write_text(json.dumps(settings, indent=2))
save_settings()
def worker(action):
    logfile=OUTPUT/(action+'-console.log')
    with logfile.open('a') as log:
        p=subprocess.Popen(['node','scripts/colab_suntzu_render.mjs',action,str(CONFIG)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in p.stdout:
                print(line,end=''); log.write(line); log.flush()
            status=p.wait()
        except BaseException:
            p.terminate(); p.wait(); raise
    assert status==0, f'Worker gagal ({status}); lihat {logfile}. Jangan lanjut.'
worker('preflight')
print((OUTPUT/'webgl.json').read_text())


## 4. Tes 3 detik, audio sesuai offset
Preflight memeriksa SHA/aset, GLB, timing/audio, komposisi **SunTzu 1080×1920 30fps**. T4 WebGL dianggap terbukti hanya jika renderer menunjukkan NVIDIA T4, bukan sekadar ANGLE berhasil. Jika tidak, fallback **swangle/CPU** dicatat; bisa jauh lebih lambat. Encoding H.264 memakai CPU, bukan klaim NVENC.

In [ ]:
worker('test')
from IPython.display import Video, display
display(Video(str(OUTPUT/'test.mp4'),embed=True,width=270))
print('Offset tes:', TEST_START_FRAME/30, 'detik. Tonton dan dengarkan sinkronisasi.')


## 5. Gerbang manual → render penuh resumable
**Tonton tes dahulu.** Jika hitam, GLB/tokoh hilang atau audio salah, berhenti. Tes bukan bukti semua adegan benar. Run all tidak menyetujui otomatis: wajib mengetik frasa berikut. Chunk 300 frame baru dianggap selesai setelah receipt + hash + ffprobe + decode lolos.

In [ ]:
approval=input('Setelah menonton tes saat ini, ketik RENDER SUN TZU untuk lanjut: ')
assert approval=='RENDER SUN TZU', 'Tidak disetujui; tidak render penuh.'
settings['approvedTest']=True
save_settings()
worker('full')


## 6. QA final dan unduh
Mux tanpa `-shortest`: master di-atrim sesuai timeline, mismatch awal >100ms ditolak, tidak dipad diam-diam. Periksa contact sheet + seluruh final (dialog, gerakan, framing, akhir audio). `qa.log` memuat blackdetect/freezedetect/loudness sebagai petunjuk, bukan keputusan artistik otomatis. Skrip selesai **bukan persetujuan produksi**.

In [ ]:
from IPython.display import Image
assert (OUTPUT/'final-receipt.json').exists(), 'Full render belum berhasil.'
display(Image(filename=str(OUTPUT/'contact-sheet.jpg')))
print((OUTPUT/'final-receipt.json').read_text())
print((OUTPUT/'qa.log').read_text()[-12000:])
print('Tonton final seluruhnya di Drive:', OUTPUT/'sun-tzu.mp4')


In [ ]:
from google.colab import files
files.download(str(OUTPUT/'sun-tzu.mp4'))


In [ ]:
import zipfile
archive=pathlib.Path('/content/suntzu-render-logs.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.iterdir():
        if p.suffix in ['.json','.log','.jpg','.txt']: z.write(p,p.name)
files.download(str(archive))


### Gagal / sesi habis
- T4 tak tersedia: jangan membeli otomatis; tunggu kuota, atau pilih CPU dan set `REQUIRE_T4=False` secara sadar.
- OOM: hentikan proses, concurrency 1, ulang preflight+tes+gate. Setting berubah berarti chunk lama dibangun ulang.
- Terputus: ulang setup pada SHA+setting sama, Drive sama. Ulang preflight/tes/gate; full memvalidasi dan melanjutkan chunk yang cocok. Jangan hapus output Drive.
- Missing entry/audio/timing/model: snapshot belum lengkap, minta SHA integrasi baru. Jangan mengganti dengan preview.
- Tidak ada klaim kecepatan sebelum tes runtime Anda sendiri.